# Brain-JEPA: single-record frozen fMRI pilot

Pinned author code: b1dfa93eb331d1e70476446b06fc8a1ac3c92345. The original environment pins need an independently prepared compatible Python/CUDA environment (candidate: Python 3.10); do not assume the default Colab interpreter is compatible. Confirm the 450-ROI order, atlas, gradient, normalization and pooling before extraction.

This notebook has NOT been executed on real MRI. Empty paths and unconfirmed recipes intentionally block inference.

In [ ]:
from pathlib import Path
import subprocess, sys
CODE_ROOT = Path('/content/MRI-classification')  # unzip the prepared code package here
assert (CODE_ROOT / 'scripts/run_features.py').is_file(), 'Upload the new local code package first; the current remote branch does not yet contain these scripts.'
PATHS = CODE_ROOT / 'configs/paths.local.yaml'
CONFIG = CODE_ROOT / 'configs/pilot.local.yaml'
HANDOFF = None  # absolute path to the actual QC-approved handoff.json


## Isolated environment
Set CREATE_ENVIRONMENT=True only when you want to install the candidate environment. Weights are acquired separately from the author; no MRI or weight download is performed by these cells.

In [ ]:
CREATE_ENVIRONMENT = False
ENV_DIR = Path('/content/venvs/jepa')
PYTHON = ENV_DIR / 'bin/python'
if CREATE_ENVIRONMENT:
    if not PYTHON.exists():
        subprocess.run([sys.executable, '-m', 'venv', str(ENV_DIR)], check=True)
    subprocess.run([str(PYTHON), '-m', 'pip', 'install', '-r', str(CODE_ROOT / 'envs/brain_jepa.txt')], check=True)
assert PYTHON.is_file(), 'Prepare/select the independent environment and set PYTHON.'


## Configure and check
Copy paths.example.yaml and pilot.yaml to the ignored *.local.yaml files, set real paths/hashes, and confirm the recipe with your teammate. A configured preflight is not proof of real model compatibility.

In [ ]:
assert PATHS.is_file() and CONFIG.is_file(), 'Create the private configuration files first.'
subprocess.run([str(PYTHON), str(CODE_ROOT/'scripts/preflight.py'), '--encoder', 'brain_jepa', '--paths', str(PATHS), '--config', str(CONFIG)], cwd=CODE_ROOT, check=True)


## Run one real checked handoff
Use a new feature_set_id for a changed recipe. Existing component outputs are never overwritten. Logs include failures. The source NIfTI/ROI array is unchanged.

In [ ]:
assert HANDOFF is not None and Path(HANDOFF).is_file(), 'Provide the real QC-approved handoff.'
DEVICE = 'cuda'
subprocess.run([str(PYTHON), str(CODE_ROOT/'scripts/run_features.py'), 'jepa', '--paths', str(PATHS), '--config', str(CONFIG), '--handoff', str(HANDOFF), '--device', DEVICE], cwd=CODE_ROOT, check=True)
